<a href="https://colab.research.google.com/github/Imesh-Isuranga/Statistical-Learning-e20154/blob/main/Bayesian%20Inference%20as%20a%20Modeling%20Framework/Answers_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses



An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


# Answers

## Task 1: Visualizing the Mechanics

The 2PL item response function is

$$
p_i(\theta) = P(Y_i = 1 \mid \Theta = \theta) = \frac{1}{1+e^{-a_i(\theta - b_i)}}.
$$

- $a_i$ (discrimination) controls the **steepness** of the curve — how sharply the probability of a correct response rises with ability.
- $b_i$ (difficulty) controls the **horizontal location** of the curve — the ability level at which $p_i(\theta) = 0.5$.

Below we plot the curve for a low discrimination item ($a=1.0$) at three difficulty levels ($b=-1,0,1$), and for comparison a high discrimination item ($a=3.0$) at $b=0$.


In [1]:
import numpy as np
import plotly.graph_objects as go

def p_2pl(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

theta = np.linspace(-6, 6, 500)

a_low = 1.0
b_values = [-1, 0, 1]

a_high = 3.0
b_high = 0

fig = go.Figure()

for b in b_values:
    fig.add_trace(go.Scatter(
        x=theta, y=p_2pl(theta, a_low, b),
        mode="lines",
        name=f"a={a_low}, b={b}"
    ))

fig.add_trace(go.Scatter(
    x=theta, y=p_2pl(theta, a_high, b_high),
    mode="lines",
    line=dict(dash="dash", width=3),
    name=f"a={a_high}, b={b_high}"
))

fig.update_layout(
    title="2PL Item Response Curves: Effect of Discrimination (a) and Difficulty (b)",
    xaxis_title="Ability θ",
    yaxis_title="P(Y=1 | Θ=θ)",
    template="plotly_white"
)
fig.show()


## Task 2: Sequential Likelihood Contribution

For a single new response $y_k$ at step $k$, conditional on $\Theta = \theta$, the response is Bernoulli with success probability $p_k(\theta)$. Hence the likelihood contribution of that single observation is

$$
L(y_k \mid \theta) = p_k(\theta)^{y_k}\big(1-p_k(\theta)\big)^{1-y_k}.
$$

Assuming conditional independence of responses given $\Theta=\theta$, the **joint likelihood** of the entire running history $\mathbf y^{(k)} = (y_1,\dots,y_k)$ is the product of the individual contributions:

$$
L\big(\mathbf y^{(k)} \mid \theta\big) = \prod_{i=1}^{k} p_i(\theta)^{y_i}\big(1-p_i(\theta)\big)^{1-y_i}.
$$

Note this joint likelihood can also be written recursively as

$$
L\big(\mathbf y^{(k)} \mid \theta\big) = L\big(\mathbf y^{(k-1)} \mid \theta\big)\cdot L(y_k \mid \theta),
$$

which is exactly what makes the *sequential* (online) update possible.


## Task 3: Mathematical Formulation of the Running Update

Since the posterior at step $k-1$, $f_{\Theta\mid \mathbf Y^{(k-1)}}(\theta \mid \mathbf y^{(k-1)})$, becomes the **prior** for step $k$, Bayes' rule applied to the single new observation $y_k$ gives

$$
f_{\Theta \mid \mathbf Y^{(k)}}\big(\theta \mid \mathbf y^{(k)}\big)
\;\propto\;
L(y_k \mid \theta)\; f_{\Theta \mid \mathbf Y^{(k-1)}}\big(\theta \mid \mathbf y^{(k-1)}\big),
$$

i.e.

$$
f_{\Theta \mid \mathbf Y^{(k)}}\big(\theta \mid \mathbf y^{(k)}\big)
\;\propto\;
p_k(\theta)^{y_k}\big(1-p_k(\theta)\big)^{1-y_k}\; f_{\Theta \mid \mathbf Y^{(k-1)}}\big(\theta \mid \mathbf y^{(k-1)}\big).
$$

The proportionality constant is the normalizing integral

$$
\int_{-\infty}^{\infty} p_k(\theta)^{y_k}\big(1-p_k(\theta)\big)^{1-y_k}\; f_{\Theta \mid \mathbf Y^{(k-1)}}\big(\theta \mid \mathbf y^{(k-1)}\big)\, d\theta,
$$

so that $f_{\Theta \mid \mathbf Y^{(k)}}$ integrates to 1. This recursion, starting from $f_\Theta^{(0)}(\theta) = \mathscr N(\theta; 0,1)$, is exactly what is implemented numerically in Task 6/7.


## Task 4: Dynamic Shifting

A correct response ($y_k=1$) enters the update as a multiplicative factor $p_k(\theta)$, which is an **increasing function of $\theta$** — it is small for $\theta \ll b_k$ and grows toward 1 for $\theta \gg b_k$, transitioning around $\theta = b_k$.

So multiplying the previous posterior by $p_k(\theta)$ **down-weights** low values of $\theta$ (those well below $b_k$) relative to high values of $\theta$. This shifts the peak (mode/mean) of the posterior **to the right**, i.e., toward larger ability values.

The size of this rightward shift depends on $b_k$: if $b_k$ is **large** (a hard item) and the user still answers correctly, this is strong evidence that $\theta$ must be large enough to clear that difficulty — so the shift in the posterior peak is **larger** than it would be from a correct answer on an easy item (small $b_k$), where getting it right is only weak evidence of high ability since even moderate-ability users would likely succeed anyway.

(Symmetrically, an incorrect answer, $y_k=0$, shifts the posterior peak to the **left**, and the size of that leftward shift is largest when the item was easy, i.e., $b_k$ small.)


## Task 5: Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls how steeply $p_k(\theta)$ transitions near $\theta = b_k$, which controls **how informative** the response is about $\theta$.

- **Large $a_k$:** The likelihood factor $p_k(\theta)^{y_k}(1-p_k(\theta))^{1-y_k}$ is close to a step function — it is very close to 1 on one side of $b_k$ and very close to 0 on the other. Multiplying the prior by such a sharply-changing function **strongly concentrates** the posterior mass near the region consistent with the observed response, producing a large **reduction in posterior variance** (the running posterior gets noticeably narrower/"sharper" after this update).

- **Small $a_k$:** The curve $p_k(\theta)$ is nearly flat (close to $0.5$ everywhere), so the likelihood factor is nearly constant in $\theta$. Multiplying the prior by an almost-constant function barely changes its shape — the posterior after the update looks almost the same as before, meaning the update contributes **little new information** and variance barely shrinks.

In short: $a_k$ governs *how much a single response can move and sharpen our belief*, while $b_k$ governs *in which direction and around which ability level* that update is centered.


## Task 6: Numerical Implementation of a Running Grid

**Algorithmic approach:**

1. **Discretize** the ability space onto a fixed, sufficiently wide and fine grid, e.g. `theta_grid = np.linspace(-6, 6, 2000)`.
2. **Initialize** the prior density values on the grid using the standard normal pdf: `prior_vals = norm.pdf(theta_grid, 0, 1)`.
3. **At each step $k$** (after observing $y_k$ with known $a_k, b_k$):
   - Compute $p_k(\theta)$ at every grid point.
   - Compute the likelihood vector $L_k(\theta) = p_k(\theta)^{y_k}(1-p_k(\theta))^{1-y_k}$ at every grid point.
   - Form the **unnormalized posterior**: `unnorm_post = prior_vals * L_k`.
   - **Normalize** so the density integrates to 1 over the grid, using numerical integration (trapezoidal rule):
     ```python
     Z = np.trapz(unnorm_post, theta_grid)   # normalizing constant
     posterior_vals = unnorm_post / Z
     ```
4. **Update state:** set `prior_vals = posterior_vals` (the posterior becomes the prior for step $k+1$), and repeat.
5. From `posterior_vals`, compute running summaries:
   - **Posterior mean:** `np.trapz(theta_grid * posterior_vals, theta_grid)`
   - **MAP estimate:** `theta_grid[np.argmax(posterior_vals)]`

This turns the abstract recursive Bayes update from Task 3 into a simple, fully numerical procedure that requires no closed-form solution — it works for any item response model, not just the 2PL.


## Task 7: Evaluating Convergence over the Timeline

We simulate a user with true ability $\theta_{\text{true}} = 0.75$ answering $n=20$ items with randomly generated difficulty and discrimination parameters, and track how the running Bayesian estimators converge toward the truth.


In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------
np.random.seed(42)

theta_true = 0.75
n_items = 20

theta_grid = np.linspace(-6, 6, 2000)

# Initialize prior: Theta ~ N(0,1)
posterior_vals = norm.pdf(theta_grid, 0, 1)

def p_2pl(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

# Storage for running estimators (include step 0 = prior, before any data)
posterior_mean_history = [np.trapz(theta_grid * posterior_vals, theta_grid)]
map_history = [theta_grid[np.argmax(posterior_vals)]]

# ---------------------------------------------------------
# Sequential simulation and Bayesian updating
# ---------------------------------------------------------
for k in range(1, n_items + 1):
    # Randomly generate item parameters
    b_k = np.random.normal(0, 1)
    a_k = np.random.uniform(0.5, 2.0)

    # True response probability at the TRUE ability
    true_prob = p_2pl(theta_true, a_k, b_k)

    # Simulate the response
    y_k = 1 if np.random.uniform(0, 1) < true_prob else 0

    # Likelihood evaluated across the grid
    p_grid = p_2pl(theta_grid, a_k, b_k)
    likelihood = p_grid**y_k * (1 - p_grid)**(1 - y_k)

    # Bayesian update
    unnorm_post = posterior_vals * likelihood
    Z = np.trapz(unnorm_post, theta_grid)
    posterior_vals = unnorm_post / Z

    # Track running estimators
    posterior_mean_history.append(np.trapz(theta_grid * posterior_vals, theta_grid))
    map_history.append(theta_grid[np.argmax(posterior_vals)])

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
steps = list(range(0, n_items + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=posterior_mean_history,
    mode="lines+markers",
    name="Posterior Mean (Bayes estimate)"
))

fig.add_trace(go.Scatter(
    x=steps, y=map_history,
    mode="lines+markers",
    name="MAP estimate"
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="black",
    annotation_text=f"θ_true = {theta_true}",
    annotation_position="bottom right"
)

fig.update_layout(
    title="Convergence of Running Bayesian Estimators to True Ability",
    xaxis_title="Item number (k)",
    yaxis_title="Estimated θ",
    template="plotly_white"
)
fig.show()


/tmp/ipykernel_836/3144792389.py:22: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_836/3144792389.py:45: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_836/3144792389.py:49: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



**Analysis.**

As $k$ increases, both the running posterior mean and the running MAP estimate generally move **closer** to $\theta_{\text{true}} = 0.75$, though the path is not perfectly monotonic early on, with very little data, a single unlucky or lucky response (especially from a high-discrimination item) can swing the estimate noticeably, since the prior is still weak and each new likelihood factor has a large relative effect on the posterior.

As more items accumulate, the posterior becomes progressively **narrower** (each update multiplies in another informative likelihood factor), so later observations move the estimate by smaller and smaller amounts the estimators "settle" toward $\theta_{\text{true}}$ and fluctuate less. The posterior mean and the MAP estimate also converge toward each other as the posterior becomes more sharply peaked and closer to symmetric (Gaussian-like) around its mode.

This illustrates the platform's growing **confidence** in its ability measurement: with few responses, ability estimates are unstable and heavily influenced by the prior; with many responses, the estimate stabilizes near the truth and the platform can be increasingly confident in it exactly the general Bayesian principle that posterior uncertainty shrinks as evidence accumulates.


# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

# Answers

## Task 1: Structural Probability and Properties

The Beta density is

$$
f_\Theta(\theta) = \frac{1}{\mathrm{B}(\alpha,\beta)}\,\theta^{\alpha-1}(1-\theta)^{\beta-1}, \qquad 0<\theta<1.
$$

We plot it for three parameter pairs: an uninformative state $(1,1)$, a right-skewed (low-CTR-favoring) state $(2,8)$, and a left-skewed (high-CTR-favoring) state $(8,2)$.


In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta as beta_dist

theta = np.linspace(0, 1, 500)

param_sets = [
    (1, 1, "Uninformative: Beta(1,1)"),
    (2, 8, "Right-skewed: Beta(2,8)"),
    (8, 2, "Left-skewed: Beta(8,2)"),
]

fig = go.Figure()
for a, b, label in param_sets:
    fig.add_trace(go.Scatter(
        x=theta, y=beta_dist.pdf(theta, a, b),
        mode="lines", name=label
    ))

fig.update_layout(
    title="Beta(α, β) Density for Different Parameter Pairs",
    xaxis_title="θ (click-through rate)",
    yaxis_title="Density",
    template="plotly_white"
)
fig.show()


**Interpretation.**
The mean of a $\text{Beta}(\alpha,\beta)$ distribution is $\dfrac{\alpha}{\alpha+\beta}$, so the **balance** between $\alpha$ and $\beta$ — not their absolute size — determines where the density's center of mass sits on $[0,1]$.

- $(\alpha=1,\beta=1)$: mean $=0.5$, flat and uninformative — every value of $\theta$ is equally plausible a priori.
- $(\alpha=2,\beta=8)$: mean $=0.2$, mass concentrated toward **low** $\theta$ — encodes a prior belief that the ad's CTR is probably low.
- $(\alpha=8,\beta=2)$: mean $=0.8$, mass concentrated toward **high** $\theta$ — encodes a prior belief that the CTR is probably high.

Increasing $\alpha$ relative to $\beta$ pulls the density mass to the right (toward 1); increasing $\beta$ relative to $\alpha$ pulls it to the left (toward 0). The **sum** $\alpha+\beta$ additionally controls how *peaked/concentrated* the density is around that mean (larger sum = more concentrated = stronger prior conviction).


## Task 2: Sequential Likelihood and Joint History

Conditional on $\Theta=\theta$, a single interaction $Y_k$ is Bernoulli($\theta$), so its likelihood contribution is

$$
L(y_k \mid \theta) = \theta^{y_k}(1-\theta)^{1-y_k}.
$$

Assuming conditional independence of interactions given $\Theta=\theta$, the joint likelihood of the running history $\mathbf y^{(k)}=(y_1,\dots,y_k)$ is

$$
L\big(\mathbf y^{(k)} \mid \theta\big) = \prod_{i=1}^{k}\theta^{y_i}(1-\theta)^{1-y_i}
= \theta^{\sum_{i=1}^k y_i}(1-\theta)^{k-\sum_{i=1}^k y_i}.
$$

Writing $h_k = \sum_{i=1}^k y_i$ (running number of clicks) and $t_k = k - h_k$ (running number of non-clicks), this simplifies to

$$
L\big(\mathbf y^{(k)} \mid \theta\big) = \theta^{h_k}(1-\theta)^{t_k}.
$$


## Task 3: Closed-Form Analytical Updates (Conjugacy)

The posterior at step $k-1$ acts as the prior for step $k$, so applying Bayes' rule to the single new observation $y_k$:

$$
f_{\Theta \mid \mathbf Y^{(k)}}\big(\theta \mid \mathbf y^{(k)}\big)
\;\propto\;
L(y_k \mid \theta)\; f_{\Theta \mid \mathbf Y^{(k-1)}}\big(\theta \mid \mathbf y^{(k-1)}\big).
$$

Suppose (induction hypothesis) that the prior at step $k-1$ is $\text{Beta}(\alpha_{k-1},\beta_{k-1})$:

$$
f_{\Theta \mid \mathbf Y^{(k-1)}}(\theta) \propto \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}.
$$

Multiplying by the new likelihood factor $\theta^{y_k}(1-\theta)^{1-y_k}$:

$$
f_{\Theta \mid \mathbf Y^{(k)}}(\theta) \;\propto\; \theta^{\alpha_{k-1}+y_k-1}(1-\theta)^{\beta_{k-1}+(1-y_k)-1}.
$$

This has exactly the algebraic form of a $\text{Beta}(\alpha_k,\beta_k)$ kernel, which **proves the Beta family is closed under this update (Beta–Binomial conjugacy)**, with

$$
\boxed{\alpha_k = \alpha_{k-1} + y_k, \qquad \beta_k = \beta_{k-1} + (1-y_k).}
$$

In words: a click ($y_k=1$) increments $\alpha$ by 1; a non-click ($y_k=0$) increments $\beta$ by 1. Unrolled over all $k$ steps starting from $(\alpha_0,\beta_0)$:

$$
\alpha_k = \alpha_0 + h_k, \qquad \beta_k = \beta_0 + t_k,
$$

where $h_k,t_k$ are the running click/non-click counts from Task 2 — matching the standard Beta-Binomial posterior.

The **posterior mean** at step $k$ is therefore the simple closed-form expression

$$
\mathbb E\big[\Theta \mid \mathbf Y^{(k)}=\mathbf y^{(k)}\big] = \frac{\alpha_k}{\alpha_k+\beta_k}.
$$


## Task 4: Dynamic Shifting Mechanics

Since $\alpha_k$ is incremented only by clicks and $\beta_k$ only by non-clicks:

- **A click ($y_k=1$)** increases $\alpha_k$ by 1, which increases the posterior mean $\dfrac{\alpha_k}{\alpha_k+\beta_k}$ and shifts the peak of the density **rightward** (toward higher CTR).
- **A non-click ($y_k=0$)** increases $\beta_k$ by 1, which decreases the posterior mean and shifts the peak **leftward** (toward lower CTR).

The size of each shift naturally shrinks as $k$ grows, since $\alpha_k+\beta_k$ (the "effective sample size") grows with every observation, so each individual $+1$ increment has a proportionally smaller effect on the mean.

**Contrast with non-conjugate models (e.g. 2PL IRT).** Here, because the Beta prior is *conjugate* to the Bernoulli/Binomial likelihood, the posterior update reduces to **simple integer arithmetic** on $(\alpha,\beta)$ — no integration is ever required, and the posterior density is known in exact closed form at every step.

In the 2PL item-response setting, by contrast, the logistic likelihood $p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}}$ is **not conjugate** to the Gaussian (or any standard) prior — multiplying them does not yield a density from a known parametric family. There, the only way to obtain the updated posterior is to numerically evaluate the product of prior and likelihood on a discretized grid of $\theta$ values and renormalize by numerical integration (e.g. the trapezoidal rule) at every step. The Beta-Binomial case is a special, computationally cheap exception to that general rule.


## Task 5: Running Point Estimators

Directly from the updated shape parameters $\alpha_k,\beta_k$:

**Running Posterior Mean:**

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)} = \mathbb E[\Theta\mid \mathbf Y^{(k)}] = \frac{\alpha_k}{\alpha_k+\beta_k}.
$$

**Running MAP estimate** (mode of the Beta density, valid when $\alpha_k>1$ and $\beta_k>1$):

$$
\widehat\theta_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k+\beta_k-2}.
$$

(Edge cases: if $\alpha_k=\beta_k=1$ the density is flat/uniform and the mode is not unique; if $\alpha_k<1$ or $\beta_k<1$ the density is unbounded and the mode occurs at a boundary, $\theta=0$ or $\theta=1$. In the simulation below, since $\alpha_0=\beta_0=1$ and $\alpha_k,\beta_k$ only increase, this degenerate case only matters at $k=0$.)


## Task 6: Performance Tracking and Convergence Analysis

We simulate an ad with true hidden CTR $\theta_{\text{true}}=0.35$ over $n=100$ impressions, starting from an uninformative $\text{Beta}(1,1)$ prior, and track the closed-form running estimators.


In [4]:
import numpy as np
import plotly.graph_objects as go

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------
np.random.seed(42)

theta_true = 0.35
n_impressions = 100

alpha_0, beta_0 = 1, 1
alpha_k, beta_k = alpha_0, beta_0

# Storage for running estimators (include step 0 = prior, before any data)
posterior_mean_history = [alpha_k / (alpha_k + beta_k)]

def map_estimate(a, b):
    if a > 1 and b > 1:
        return (a - 1) / (a + b - 2)
    return a / (a + b)  # fallback to mean when mode is undefined/at boundary

map_history = [map_estimate(alpha_k, beta_k)]

# ---------------------------------------------------------
# Sequential simulation and closed-form Bayesian updating
# ---------------------------------------------------------
for k in range(1, n_impressions + 1):
    # Simulate the response
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Closed-form conjugate update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Track running estimators
    posterior_mean_history.append(alpha_k / (alpha_k + beta_k))
    map_history.append(map_estimate(alpha_k, beta_k))

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
steps = list(range(0, n_impressions + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=posterior_mean_history,
    mode="lines", name="Posterior Mean (Bayes estimate)"
))

fig.add_trace(go.Scatter(
    x=steps, y=map_history,
    mode="lines", name="MAP estimate"
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="black",
    annotation_text=f"θ_true = {theta_true}",
    annotation_position="bottom right"
)

fig.update_layout(
    title="Convergence of Running Beta-Binomial Estimators to True CTR",
    xaxis_title="Impression number (k)",
    yaxis_title="Estimated θ",
    template="plotly_white"
)
fig.show()


**Analysis.**

Early on (small $k$), $\alpha_k+\beta_k$ is small, so each single impression carries substantial weight the running posterior mean and MAP can swing noticeably after just one or two clicks/non-clicks, and the estimators may sit relatively far from $\theta_{\text{true}}=0.35$.

As $k \to 100$, the effective sample size $\alpha_k+\beta_k = \alpha_0+\beta_0+k$ grows large, so each new $+1$ increment changes the ratio $\dfrac{\alpha_k}{\alpha_k+\beta_k}$ by a progressively smaller amount. The running estimators therefore **stabilize and converge** toward $\theta_{\text{true}}$, with the posterior mean and MAP estimate also converging to each other as the Beta distribution becomes increasingly peaked and closer to symmetric.

This demonstrates the general Bayesian principle that **the influence of the prior is overwhelmed by accumulating data**: with the uninformative $\text{Beta}(1,1)$ start, the initial belief contributes only "2 pseudo-observations" worth of information ($\alpha_0+\beta_0=2$), which becomes negligible once $k$ reaches 100 real observations. Had a more informative (larger $\alpha_0+\beta_0$) prior been chosen instead, convergence to $\theta_{\text{true}}$ would take proportionally longer, since the prior's pseudo-count would need to be "diluted" by a larger number of real impressions before the data dominates.


# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

# Answers

## Task 1: Prior Belief Boundaries

The initial prior is $\Theta \sim \text{Beta}(8, 1.5)$ on the physical domain $\theta\in(0,1]$:

$$
f_\Theta^{(0)}(\theta) = \frac{1}{\mathrm B(8,1.5)}\,\theta^{7}(1-\theta)^{0.5}.
$$


In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta as beta_dist

alpha0, beta0 = 8, 1.5
theta = np.linspace(0.01, 1.0, 500)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theta, y=beta_dist.pdf(theta, alpha0, beta0),
    mode="lines", name=f"Prior: Beta({alpha0},{beta0})"
))
fig.update_layout(
    title="Initial Prior Belief: Stiffness Efficiency Factor θ",
    xaxis_title="θ (stiffness efficiency)",
    yaxis_title="Density",
    template="plotly_white"
)
fig.show()

prior_mean = alpha0 / (alpha0 + beta0)
print(f"E[Theta^(0)] = {alpha0}/({alpha0}+{beta0}) = {prior_mean:.4f}")


E[Theta^(0)] = 8/(8+1.5) = 0.8421


**Analytical expected prior stiffness:**

$$
\mathbb E[\Theta^{(0)}] = \frac{\alpha_0}{\alpha_0+\beta_0} = \frac{8}{8+1.5} = \frac{8}{9.5} \approx 0.842.
$$

**Why is $\text{Beta}(8,1.5)$ an appropriate initial prior?** Since $\alpha_0=8 \gg \beta_0=1.5$, the density is strongly **left-skewed toward 1**: nearly all of its probability mass sits close to $\theta=1$ (pristine condition), with a rapidly vanishing tail as $\theta\to 0$. This matches the engineering assumption that a newly instrumented, recently-inspected component is *very likely* healthy — the prior encodes "probably close to 100% stiffness, but with some allowance for minor pre-existing wear" — while still assigning nonzero (if small) probability to lower stiffness values, so the model can be updated by evidence rather than ruling out damage a priori.


## Task 2: Structural Likelihood Formulation

The measurement model is $y_k = \theta\, K_{\text{nominal}}\, e^{\epsilon_k}$ with $\epsilon_k \sim \mathscr N(0,\sigma^2)$. Taking logarithms,

$$
\ln y_k = \ln\theta + \ln K_{\text{nominal}} + \epsilon_k
\quad\Longrightarrow\quad
\ln y_k \mid \Theta=\theta \;\sim\; \mathscr N\big(\ln(\theta K_{\text{nominal}}),\ \sigma^2\big).
$$

This means, conditional on $\Theta=\theta$, $Y_k$ follows a **log-normal distribution** with location parameter $\ln(\theta K_{\text{nominal}})$ and scale $\sigma$. Using the log-normal density formula, the likelihood contribution of a single measurement $y_k$ is

$$
L(y_k\mid\theta) = f_{Y_k\mid\Theta}(y_k\mid\theta) = \frac{1}{y_k\,\sigma\sqrt{2\pi}}\exp\left[-\frac{\big(\ln y_k - \ln(\theta K_{\text{nominal}})\big)^2}{2\sigma^2}\right].
$$

(Since the factor $\frac{1}{y_k\sigma\sqrt{2\pi}}$ does not depend on $\theta$, it acts only as a $\theta$-independent multiplicative constant when this expression is used as a likelihood *function of $\theta$* — it cancels in the normalization step but is included above for completeness as a proper density.)

Assuming conditional independence of measurements given $\Theta=\theta$, the joint likelihood of the running history $\mathbf y^{(k)}=(y_1,\dots,y_k)$ is

$$
L\big(\mathbf y^{(k)}\mid\theta\big) = \prod_{i=1}^{k}\frac{1}{y_i\sigma\sqrt{2\pi}}\exp\left[-\frac{\big(\ln y_i - \ln(\theta K_{\text{nominal}})\big)^2}{2\sigma^2}\right].
$$


## Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

**Why there is no closed-form conjugate solution here.** The Beta prior density has the algebraic form $\theta^{\alpha-1}(1-\theta)^{\beta-1}$, a *polynomial-type* kernel in $\theta$. The log-normal likelihood, however, is an exponential of a **quadratic function of $\ln\theta$** (not of $\theta$ itself):

$$
L(y_k\mid\theta) \propto \exp\left[-\frac{(\ln\theta - c_k)^2}{2\sigma^2}\right], \qquad c_k \triangleq \ln y_k - \ln K_{\text{nominal}}.
$$

Multiplying a polynomial-in-$\theta$ kernel by an exponential-quadratic-in-$\ln\theta$ kernel does **not** produce a density belonging to any standard named parametric family (it is neither Beta, nor log-normal, nor any simple combination) — there is no algebraic simplification that collapses the product back into a form with a fixed, small number of updatable parameters. Hence the posterior must be tracked as an arbitrary function on a grid rather than through a handful of closed-form parameter updates (unlike the Beta–Binomial case).

**Recursive relationship (up to proportionality):**

$$
f_{\Theta\mid \mathbf Y^{(k)}}\big(\theta\mid \mathbf y^{(k)}\big) \;\propto\; L(y_k\mid\theta)\; f_{\Theta\mid \mathbf Y^{(k-1)}}\big(\theta\mid \mathbf y^{(k-1)}\big), \qquad \theta\in(0,1],
$$

with the proportionality constant given by the normalizing integral over the bounded domain,

$$
\int_{0}^{1} L(y_k\mid\theta)\, f_{\Theta\mid \mathbf Y^{(k-1)}}\big(\theta\mid \mathbf y^{(k-1)}\big)\, d\theta.
$$


## Task 4: Running Point Estimates

**Running Posterior Mean** — a weighted average of $\theta$ over the bounded domain, weighted by the current posterior density:

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)} = \mathbb E\big[\Theta \mid \mathbf Y^{(k)}=\mathbf y^{(k)}\big] = \int_{0}^{1} \theta\, f_{\Theta\mid \mathbf Y^{(k)}}\big(\theta\mid\mathbf y^{(k)}\big)\, d\theta.
$$

**Running MAP estimate** — the location of the mode of the posterior density. This is **not** itself expressed as an integral (it is an optimization, not an average); however, evaluating it requires the posterior density to already be correctly normalized, which *does* require the integral above (as a normalizing constant) to have been computed first:

$$
\widehat\theta_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0,1]} f_{\Theta\mid \mathbf Y^{(k)}}\big(\theta\mid\mathbf y^{(k)}\big).
$$

In practice, both quantities are evaluated numerically from the same discretized grid: the mean via numerical quadrature (e.g. the trapezoidal rule approximating the integral above), and the MAP via a simple `argmax` over the grid points of the already-normalized density.


## Task 5: Algorithmic Grid Approximation and Normalization

**Step-by-step numerical procedure:**

1. **Discretize the bounded domain.** Since $\theta\in(0,1]$ and the likelihood involves $\ln\theta$ (undefined at $\theta=0$), build the grid strictly away from the singular boundary, e.g. `theta_grid = np.linspace(1e-3, 1.0, 2000)` — this avoids evaluating $\ln(0)$ while still covering essentially the full physical range (the excluded sliver near 0 carries negligible prior/posterior mass anyway given the healthy-biased prior).
2. **Initialize** the prior density values on the grid: `prior_vals = beta.pdf(theta_grid, alpha0, beta0)`.
3. **At each step $k$**, given the new reading $y_k$:
   - Evaluate the likelihood at every grid point using the log-normal form from Task 2 (equivalently, `norm.pdf(np.log(y_k), loc=np.log(theta_grid * K_nominal), scale=sigma)`).
   - Form the unnormalized posterior: `unnorm_post = prior_vals * likelihood_vals`.
   - **Normalize** using the trapezoidal rule over the bounded grid:
     ```python
     Z = np.trapezoid(unnorm_post, theta_grid)
     posterior_vals = unnorm_post / Z
     ```
4. **Update state:** set `prior_vals = posterior_vals` for the next step's prior, and repeat.
5. **Extract estimates:**
   - Posterior mean: `np.trapezoid(theta_grid * posterior_vals, theta_grid)`
   - MAP: `theta_grid[np.argmax(posterior_vals)]`

**Handling the boundary.** Because $\theta=1$ (the healthy end) is a closed, reachable boundary of the physical domain, the grid should include $\theta=1.0$ exactly as its last point; the lower boundary is instead handled by starting the grid at a small $\epsilon>0$ (e.g. $10^{-3}$) rather than exactly 0, purely to keep $\ln\theta$ finite — this has no meaningful effect on the computed mean/MAP as long as the posterior density assigns negligible mass near that lower cutoff (which it will, given a healthy-biased prior and reasonable measurements).


## Task 6: Performance Tracking and Degradation Convergence Analysis

We simulate a sudden impact dropping the true stiffness to $\theta_{\text{true}}=0.68$, and track how the bounded-grid posterior reacts over $n=15$ sensor readings.


In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta as beta_dist, norm

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------
np.random.seed(42)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_steps = 15

alpha0, beta0 = 8, 1.5
theta_grid = np.linspace(1e-3, 1.0, 2000)

posterior_vals = beta_dist.pdf(theta_grid, alpha0, beta0)
posterior_vals /= np.trapezoid(posterior_vals, theta_grid)  # ensure normalized at k=0

posterior_mean_history = [np.trapezoid(theta_grid * posterior_vals, theta_grid)]
map_history = [theta_grid[np.argmax(posterior_vals)]]

milestones = {0, 1, 2, 5, 10, 15}
milestone_curves = {0: posterior_vals.copy()}

# ---------------------------------------------------------
# Sequential simulation and grid-based Bayesian updating
# ---------------------------------------------------------
for k in range(1, n_steps + 1):
    # Simulate a noisy sensor reading from the true physics model
    epsilon_k = np.random.normal(0, sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon_k)

    # Likelihood evaluated across the grid (log-normal form)
    likelihood = norm.pdf(np.log(y_k), loc=np.log(theta_grid * K_nominal), scale=sigma)

    # Bayesian update on the bounded grid
    unnorm_post = posterior_vals * likelihood
    Z = np.trapezoid(unnorm_post, theta_grid)
    posterior_vals = unnorm_post / Z

    # Track running estimators
    posterior_mean_history.append(np.trapezoid(theta_grid * posterior_vals, theta_grid))
    map_history.append(theta_grid[np.argmax(posterior_vals)])

    if k in milestones:
        milestone_curves[k] = posterior_vals.copy()

# ---------------------------------------------------------
# Plot 1: posterior density curves at milestones
# ---------------------------------------------------------
fig1 = go.Figure()
for k in sorted(milestone_curves):
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=milestone_curves[k],
        mode="lines", name=f"k={k}"
    ))
fig1.add_vline(
    x=theta_true, line_dash="dash", line_color="black",
    annotation_text=f"θ_true = {theta_true}", annotation_position="top"
)
fig1.update_layout(
    title="Evolution of the Posterior Density Across Inspection Milestones",
    xaxis_title="θ (stiffness efficiency)",
    yaxis_title="Density",
    template="plotly_white"
)
fig1.show()

# ---------------------------------------------------------
# Plot 2: convergence of running estimators
# ---------------------------------------------------------
steps = list(range(0, n_steps + 1))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=steps, y=posterior_mean_history,
    mode="lines+markers", name="Posterior Mean (Bayes estimate)"
))
fig2.add_trace(go.Scatter(
    x=steps, y=map_history,
    mode="lines+markers", name="MAP estimate"
))
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="black",
    annotation_text=f"θ_true = {theta_true}", annotation_position="bottom right"
)
fig2.update_layout(
    title="Convergence of Running Estimators to True Remaining Stiffness",
    xaxis_title="Inspection step (k)",
    yaxis_title="Estimated θ",
    template="plotly_white"
)
fig2.show()


**Analysis.**

At $k=0$, the posterior is just the optimistic prior, concentrated near $\theta\approx 0.84$ — far above the true $\theta_{\text{true}}=0.68$. As the first few noisy readings arrive, the likelihood (centered near $0.68$) begins pulling the posterior mass leftward, and the running mean/MAP estimates visibly move away from the prior toward the true value. Because $\sigma=0.15$ produces moderately noisy individual readings, the first one or two measurements alone are usually not enough to fully overcome the strong healthy-biased prior — it typically takes on the order of **several readings (often somewhere around $k\approx 3$–$6$, depending on the particular noise realization)** before the accumulated evidence decisively outweighs the prior's initial pull toward $\theta=1$, after which the estimators settle close to $0.68$ and change only slightly with each additional reading.

The posterior density curves at the milestones show this directly: the $k=0$ curve is tall and narrow around $\approx 0.84$; intermediate curves ($k=1,2$) are wider and possibly still somewhat biased high or bimodal-looking as prior and data compete; by $k=5,10,15$ the curve has not only shifted to be centered near $0.68$ but has also become progressively **narrower**, reflecting shrinking posterior variance as more evidence accumulates.

**Implication for structural safety.** The narrowing of the posterior directly reflects increasing *confidence* in the damage estimate — which matters practically because a safety threshold decision (e.g. "flag for maintenance if $\theta$ is below some critical value with high confidence") requires not just a point estimate but a **tight enough distribution** around it. A wide posterior early on could still assign non-trivial probability to $\theta$ being either safely high or dangerously low, making any single threshold-crossing alert unreliable; only once the posterior has sufficiently narrowed (after enough sensor readings) can engineers trust that the estimated 68% stiffness state reflects the true structural condition with the statistical confidence needed to trigger a maintenance response.


# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

# Answers

## Part 1: Deriving the Marginal Density

By the law of total probability, we condition on the latent cluster label $C_i$ and sum over all $K$ possible values:

$$
p(x_i) = \sum_{k=1}^K P(C_i=k)\, p(x_i \mid C_i=k).
$$

By assumption, $P(C_i=k) = \phi_k$ and, conditional on $C_i=k$, $X_i \sim \mathscr N(\mu_k,\Sigma_k)$, so $p(x_i\mid C_i=k) = \mathscr N(x_i\mid \mu_k,\Sigma_k)$. Substituting:

$$
\boxed{p(x_i) = \sum_{k=1}^K \phi_k\, \mathscr N(x_i \mid \mu_k,\Sigma_k).}
$$

**Why "mixture" density?** This is a convex combination (weighted average, with non-negative weights summing to 1) of $K$ Gaussian "component" densities. It is not itself Gaussian in general — it can be multimodal, skewed, or heavy-tailed — because it describes a population that is a **mixture** of $K$ distinct homogeneous subpopulations, each individually Gaussian, blended together in proportions $\phi_1,\dots,\phi_K$.


## Part 2: Deriving the Posterior Cluster Probability

Bayes' rule for the discrete latent label $C_i$ given the continuous observation $X_i=x_i$ states

$$
P(C_i=k\mid X_i=x_i) = \frac{P(X_i=x_i\mid C_i=k)\,P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i\mid C_i=j)\,P(C_i=j)}.
$$

The denominator is exactly the marginal density $p(x_i)$ derived in Part 1 (law of total probability again), which guarantees the right-hand side is a valid probability distribution over $k=1,\dots,K$ that sums to 1.

Substituting $P(C_i=k)=\phi_k$ and $P(X_i=x_i\mid C_i=k)=\mathscr N(x_i\mid\mu_k,\Sigma_k)$:

$$
\boxed{P(C_i=k\mid X_i=x_i) = \frac{\phi_k\,\mathscr N(x_i\mid \mu_k,\Sigma_k)}{\sum_{j=1}^K \phi_j\,\mathscr N(x_i\mid \mu_j,\Sigma_j)} \;\triangleq\; \gamma_{ik}.}
$$

**Why is $\gamma_{ik}$ a posterior probability?** $\phi_k$ is the *prior* belief that a randomly chosen point belongs to cluster $k$, before seeing where it lies. After observing the actual location $x_i$, we update that belief using how compatible $x_i$ is with each cluster's Gaussian shape (the likelihood term $\mathscr N(x_i\mid\mu_k,\Sigma_k)$). The resulting $\gamma_{ik}$ is exactly the Bayesian posterior — our revised belief about which cluster generated $x_i$, now that we have evidence.


## Part 3: One-Hot Encoding of the Latent Cluster Variable

By definition, $Z_{ik}\in\{0,1\}$ with $Z_{ik}=1$ iff $C_i=k$. The conditional expectation of a $\{0,1\}$-valued random variable is simply the conditional probability that it equals 1:

$$
\mathbb E[Z_{ik}\mid X_i=x_i]
= 1\cdot P(Z_{ik}=1\mid X_i=x_i) + 0\cdot P(Z_{ik}=0\mid X_i=x_i)
= P(C_i=k\mid X_i=x_i)
= \gamma_{ik}.
$$

Stacking this over $k=1,\dots,K$ componentwise:

$$
\mathbb E[Z_i \mid X_i=x_i] =
\begin{bmatrix}
\mathbb E[Z_{i1}\mid X_i=x_i]\\
\vdots\\
\mathbb E[Z_{iK}\mid X_i=x_i]
\end{bmatrix}
=
\begin{bmatrix}
\gamma_{i1}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$

**Conclusion.** The vector of responsibilities $(\gamma_{i1},\dots,\gamma_{iK})$ — the *soft cluster assignment* of $x_i$ — is nothing but the conditional expectation $\mathbb E[Z_i\mid X_i=x_i]$ of the latent one-hot membership vector, given the observed data. This is the precise measure-theoretic sense in which GMM clustering is "conditional-expectation-based clustering."


## Part 4: From Soft Assignment to Hard Clustering

**Soft clustering** retains the *full* posterior vector $\mathbb E[Z_i\mid X_i=x_i] = (\gamma_{i1},\dots,\gamma_{iK})$, expressing the observation's membership as a **probability distribution** over all $K$ clusters simultaneously — e.g. $x_i$ might be "70% cluster 1, 25% cluster 2, 5% cluster 3." This retains uncertainty, which is especially informative near cluster boundaries.

**Hard clustering** collapses this distribution to a single label by taking the most probable cluster,

$$
\widehat C_i = \arg\max_{1\le k\le K}\gamma_{ik},
$$

discarding the relative confidence and any ambiguity — every point is forced into exactly one bucket regardless of how close the runner-up cluster's responsibility was. Hard assignment is convenient for producing a final partition of the data, but it throws away the very uncertainty information that made the Bayesian/EM approach meaningful in the first place.


## Part 5: Conditional Expectation of the Observation Given the Cluster

Since $X_i \mid C_i=k \sim \mathscr N(\mu_k,\Sigma_k)$, and the mean of a multivariate Gaussian $\mathscr N(\mu_k,\Sigma_k)$ is $\mu_k$ by definition of that distribution's first moment,

$$
\boxed{\mathbb E[X_i \mid C_i=k] = \mu_k.}
$$

**Interpretation.** $\mu_k$ is the average location of all observations that truly belong to cluster $k$ — i.e., the "center of mass" or **centroid** of cluster $k$ in $\mathbb R^d$.

**Comparing the two conditional expectations:**

| | $\mathbb E[Z_i\mid X_i=x_i]$ | $\mathbb E[X_i\mid C_i=k]$ |
|---|---|---|
| Conditions on | an observed data point $x_i$ | a cluster label $k$ |
| Lives in | $\mathbb R^K$ (probability simplex) | $\mathbb R^d$ (feature space) |
| Answers | "Given this point, how likely is each cluster?" | "Given this cluster, where do its points sit on average?" |

The first expectation goes **from data to cluster membership** (a soft label for one point); the second goes **from cluster to typical data location** (a summary of an entire subpopulation). They are, in effect, expectations in "opposite directions" of the same joint model $(X_i,C_i)$.


## Part 6: The Complete-Data Likelihood

Taking logarithms of the complete-data likelihood

$$
p(x_1,\dots,x_n,z_1,\dots,z_n) = \prod_{i=1}^n\prod_{k=1}^K \left[\phi_k\,\mathscr N(x_i\mid\mu_k,\Sigma_k)\right]^{z_{ik}},
$$

and using $\log(\prod)=\sum(\log)$ together with $\log(a^{z}) = z\log a$:

$$
\ell_c = \log p(x_1,\dots,x_n,z_1,\dots,z_n)
= \sum_{i=1}^n\sum_{k=1}^K z_{ik}\Big[\log\phi_k + \log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Big].
$$

**Why is this easy to maximize if $z_{ik}$ were known?** Because $z_{ik}\in\{0,1\}$ and exactly one $z_{ik}=1$ per observation $i$ (the rest are 0), the inner sum over $k$ collapses to a single term for each $i$ — the label of the cluster that $x_i$ truly belongs to. This means $\ell_c$ decomposes into $K$ **completely separate** sub-problems: within each cluster $k$, we just fit a standard (unweighted) Gaussian MLE and a simple proportion for $\phi_k$, using only the subset of points labeled $k$. There is no coupling between clusters and no need for iterative optimization — closed-form MLE formulas apply directly, exactly as in ordinary supervised Gaussian classification.


## Part 7: The EM Interpretation

Since the true labels $z_{ik}$ are unobserved, they cannot be plugged directly into $\ell_c$. The EM algorithm instead replaces each unknown indicator by its conditional expectation given the observed data **and the current parameter estimates**:

$$
z_{ik} \;\leadsto\; \mathbb E[Z_{ik}\mid X_i=x_i] = \gamma_{ik}.
$$

Substituting this into $\ell_c$ term-by-term gives the **expected complete-data log-likelihood**:

$$
Q = \sum_{i=1}^n\sum_{k=1}^K \gamma_{ik}\Big[\log\phi_k + \log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Big].
$$

**Why is this the E-step?** Computing $\gamma_{ik}$ requires the *current* parameter estimates $(\phi_k,\mu_k,\Sigma_k)$ (from Part 2's formula) — it is literally a Bayesian posterior update of cluster membership probabilities, conditional on the current model. Nothing about the model parameters changes in this step; only our *belief* about which cluster generated each point is refreshed given where the parameters currently sit. This is precisely a conditional-expectation ("Expectation") computation, hence the name **E-step**.


## Part 8: Parameter Updates (the M-step)

We maximize $Q$ with respect to $\phi_k,\mu_k,\Sigma_k$, treating the $\gamma_{ik}$ (from the E-step) as fixed known weights.

**Mixture weights $\phi_k$.** Maximizing $\sum_i\sum_k \gamma_{ik}\log\phi_k$ subject to the constraint $\sum_k \phi_k=1$ via a Lagrange multiplier $\lambda$:

$$
\frac{\partial}{\partial \phi_k}\left[\sum_i\gamma_{ik}\log\phi_k + \lambda\Big(\sum_k \phi_k -1\Big)\right]=0
\;\;\Longrightarrow\;\;
\frac{N_k}{\phi_k}+\lambda=0,\quad N_k \triangleq \sum_{i=1}^n \gamma_{ik}.
$$

Solving gives $\phi_k = -N_k/\lambda$; summing over $k$ and using $\sum_k\phi_k=1$ and $\sum_k N_k = n$ fixes $\lambda=-n$, so

$$
\boxed{\phi_k^{\text{new}} = \frac{N_k}{n}.}
$$

**Means $\mu_k$.** Differentiating $\sum_i \gamma_{ik}\log\mathscr N(x_i\mid\mu_k,\Sigma_k)$ with respect to $\mu_k$ and setting to zero (the Gaussian log-density is quadratic in $\mu_k$, giving a weighted least-squares normal equation) yields the weighted sample mean:

$$
\boxed{\mu_k^{\text{new}} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}\, x_i.}
$$

**Covariances $\Sigma_k$.** Similarly, differentiating with respect to $\Sigma_k^{-1}$ (or $\Sigma_k$) and solving gives the weighted sample covariance about the *updated* mean:

$$
\boxed{\Sigma_k^{\text{new}} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}\,(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^T.}
$$

**Interpretation of $\gamma_{ik}$ as a fractional weight.** Unlike hard clustering (where each point contributes fully to exactly one cluster's statistics), here every observation $x_i$ contributes to **every** cluster's mean and covariance, but weighted by how responsible that cluster is for it. A point with $\gamma_{ik}=0.9$ contributes almost fully to cluster $k$'s statistics and only $0.1$-worth to the others; a point sitting ambiguously between two clusters (e.g. $\gamma_{ik}=0.5$ for two clusters) contributes half-weight to each. $N_k=\sum_i \gamma_{ik}$ is thus the *effective (fractional) number of points* assigned to cluster $k$.


## Part 9: Interpretation

Gaussian mixture clustering can be viewed as a repeated cycle of conditional updating between two spaces: cluster identity and data location. The mixture weight $\phi_k$ encodes our **prior** belief about how common cluster $k$ is, before looking at any particular point. For a specific observation $x_i$, the Gaussian density $\mathscr N(x_i\mid\mu_k,\Sigma_k)$ measures how *compatible* that point's location is with cluster $k$'s shape — essentially a likelihood. Combining the two via Bayes' rule produces the responsibility $\gamma_{ik}$, the **posterior** probability that $x_i$ belongs to cluster $k$ after actually observing where it lies; stacked across clusters, this is exactly the soft assignment vector $\mathbb E[Z_i\mid X_i=x_i]$ derived in Part 3. The **M-step** then flows information back the other way: it re-estimates each cluster's prior weight, mean, and covariance using these posterior membership probabilities as fractional weights over all the data. Repeating E-step and M-step alternately — posterior update of memberships, then re-estimation of cluster shapes given those memberships — is precisely an iterative scheme of conditional expectations feeding back into the parameters that define the next conditional expectation. In this sense, **Gaussian mixture clustering is probabilistic clustering built entirely from conditional expectations of a latent cluster membership variable**, rather than a deterministic, one-shot partitioning rule.


## Part 10: Computational Simulation and Out-of-Sample Validation


In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    # Two-dimensional Gaussian Mixture Model segmenter with train/test
    # validation and interactive Plotly diagnostics.

    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = None
        self.feature_names = None

    # -----------------------------------------------------------------
    # Data loading
    # -----------------------------------------------------------------
    def load_data(self, csv_path=None, feature_x="PURCHASES",
                  feature_y="CREDIT_LIMIT", n_synthetic=2000):
        self.feature_names = [feature_x, feature_y]

        if csv_path is not None:
            try:
                df = pd.read_csv(csv_path)
                df = df[[feature_x, feature_y]].dropna()
                self.raw_data = df.values
                print(f"Loaded {len(df)} rows from '{csv_path}'.")
                return self.raw_data
            except FileNotFoundError:
                print(f"'{csv_path}' not found — falling back to synthetic data.")

        # Synthetic fallback: three overlapping 2D Gaussian blobs
        rng = np.random.default_rng(self.random_state)
        n_per_cluster = n_synthetic // 3

        cluster_params = [
            (np.array([500, 3000]),  np.array([[40000, 5000], [5000, 300000]])),
            (np.array([2500, 7000]), np.array([[90000, 8000], [8000, 500000]])),
            (np.array([6000, 15000]), np.array([[160000, 10000], [10000, 900000]])),
        ]

        blocks = []
        for mean, cov in cluster_params:
            blocks.append(rng.multivariate_normal(mean, cov, size=n_per_cluster))
        raw = np.vstack(blocks)
        raw = np.clip(raw, 0, None)  # financial features are non-negative

        self.raw_data = raw
        print(f"Generated {len(raw)} synthetic rows (3 overlapping blobs).")
        return self.raw_data

    # -----------------------------------------------------------------
    # Preprocessing
    # -----------------------------------------------------------------
    def preprocess_and_split(self, test_size=0.2):
        X_train_raw, X_test_raw = train_test_split(
            self.raw_data, test_size=test_size, random_state=self.random_state
        )
        self.X_train_raw = X_train_raw
        self.X_test_raw = X_test_raw

        self.X_train = self.scaler.fit_transform(X_train_raw)
        self.X_test = self.scaler.transform(X_test_raw)
        return self.X_train, self.X_test

    # -----------------------------------------------------------------
    # Fitting (EM algorithm)
    # -----------------------------------------------------------------
    def fit(self):
        self.gmm = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
            max_iter=200,
        )
        self.gmm.fit(self.X_train)
        print(f"Converged: {self.gmm.converged_}")
        print(f"Iterations to converge: {self.gmm.n_iter_}")
        return self.gmm

    # -----------------------------------------------------------------
    # Out-of-sample evaluation
    # -----------------------------------------------------------------
    def evaluate(self):
        avg_ll = self.gmm.score(self.X_test)  # sklearn returns mean per-sample log-likelihood
        print(f"Average log-likelihood on held-out test set: {avg_ll:.4f}")
        return avg_ll

    # -----------------------------------------------------------------
    # Plot 1: empirical density heatmap of raw training data
    # -----------------------------------------------------------------
    def plot_density_heatmap(self):
        fig = px.density_heatmap(
            x=self.X_train_raw[:, 0], y=self.X_train_raw[:, 1],
            marginal_x="histogram", marginal_y="histogram",
            labels={"x": self.feature_names[0], "y": self.feature_names[1]},
            title="Empirical 2D Density of Training Data",
        )
        fig.show()
        return fig

    # -----------------------------------------------------------------
    # Helper: responsibility grid over feature space (in scaled units)
    # -----------------------------------------------------------------
    def _responsibility_grid(self, grid_points=200):
        x_min, x_max = self.X_train[:, 0].min() - 0.5, self.X_train[:, 0].max() + 0.5
        y_min, y_max = self.X_train[:, 1].min() - 0.5, self.X_train[:, 1].max() + 0.5

        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, grid_points),
            np.linspace(y_min, y_max, grid_points),
        )
        grid_scaled = np.column_stack([xx.ravel(), yy.ravel()])

        proba = self.gmm.predict_proba(grid_scaled)   # gamma_ik on the grid
        hard_cluster = np.argmax(proba, axis=1).reshape(xx.shape)
        max_responsibility = np.max(proba, axis=1).reshape(xx.shape)

        # Map grid back to raw (unscaled) feature units for plotting
        grid_raw = self.scaler.inverse_transform(grid_scaled)
        xx_raw = grid_raw[:, 0].reshape(xx.shape)
        yy_raw = grid_raw[:, 1].reshape(xx.shape)

        return xx_raw, yy_raw, hard_cluster, max_responsibility

    # -----------------------------------------------------------------
    # Plot 2: training assignment with responsibility contours
    # -----------------------------------------------------------------
    def plot_training_assignment(self):
        xx_raw, yy_raw, hard_cluster, _ = self._responsibility_grid()
        train_labels = self.gmm.predict(self.X_train)

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx_raw[0, :], y=yy_raw[:, 0], z=hard_cluster,
            showscale=False, opacity=0.35,
            colorscale="Portland", contours=dict(coloring="fill"),
            name="Posterior cluster region (argmax γ)"
        ))
        fig.add_trace(go.Scatter(
            x=self.X_train_raw[:, 0], y=self.X_train_raw[:, 1],
            mode="markers",
            marker=dict(color=train_labels, colorscale="Portland", size=5,
                        line=dict(width=0.5, color="black")),
            name="Training points"
        ))
        fig.update_layout(
            title="Training Data with Posterior Responsibility Contours",
            xaxis_title=self.feature_names[0], yaxis_title=self.feature_names[1],
            template="plotly_white"
        )
        fig.show()
        return fig

    # -----------------------------------------------------------------
    # Plot 3: test assignment with responsibility contours
    # -----------------------------------------------------------------
    def plot_test_assignment(self):
        xx_raw, yy_raw, hard_cluster, _ = self._responsibility_grid()
        test_labels = self.gmm.predict(self.X_test)

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx_raw[0, :], y=yy_raw[:, 0], z=hard_cluster,
            showscale=False, opacity=0.35,
            colorscale="Portland", contours=dict(coloring="fill"),
            name="Posterior cluster region (argmax γ)"
        ))
        fig.add_trace(go.Scatter(
            x=self.X_test_raw[:, 0], y=self.X_test_raw[:, 1],
            mode="markers",
            marker=dict(color=test_labels, colorscale="Portland", size=6,
                        symbol="diamond", line=dict(width=0.5, color="black")),
            name="Test points"
        ))
        fig.update_layout(
            title="Out-of-Sample Test Data with Posterior Responsibility Contours",
            xaxis_title=self.feature_names[0], yaxis_title=self.feature_names[1],
            template="plotly_white"
        )
        fig.show()
        return fig


In [9]:
# ---------------------------------------------------------
# Run the full pipeline
# ---------------------------------------------------------
segmenter = GMMFinancialSegmenter(n_components=3, random_state=42)

# Replace with the real path once you've uploaded the Kaggle CSV, e.g.:
# segmenter.load_data(csv_path="CC GENERAL.csv", feature_x="PURCHASES", feature_y="CREDIT_LIMIT")
segmenter.load_data(csv_path="CC GENERAL.csv")   # falls back to synthetic data if not found

segmenter.preprocess_and_split(test_size=0.2)
segmenter.fit()
segmenter.evaluate()

segmenter.plot_density_heatmap()
segmenter.plot_training_assignment()
segmenter.plot_test_assignment()


Loaded 8949 rows from 'CC GENERAL.csv'.
Converged: True
Iterations to converge: 18
Average log-likelihood on held-out test set: -1.6890


**Evaluating the plots.**

The **density heatmap** shows the raw, unlabeled structure of the data — if genuine multimodality exists (several distinct "blobs" of financial behavior), it appears here before any model is fit, motivating the choice of $K=3$ components.

The **training assignment plot** overlays the fitted training points on a filled contour map, where each contour region corresponds to the cluster index $k=\arg\max_k \gamma_{ik}$ evaluated on a fine grid of $(x,y)$ points spanning the feature space. This contour map is a direct, continuous visualization of the quantity proved analytically in Part 3: at *every* location in feature space (not just at the observed data points), the model computes the full soft-assignment vector $\mathbb E[Z_i\mid X_i=x_{\text{grid}}] = (\gamma_1,\dots,\gamma_K)$, and the filled color shows which component of that vector is largest. The boundaries between colors are exactly the loci where two (or more) responsibilities are tied — the points of maximal posterior ambiguity.

The **test assignment plot** repeats this using the *same* fitted contour regions (i.e., the same trained $\phi_k,\mu_k,\Sigma_k$) but overlays unseen validation points. Test points that fall near a contour boundary reveal where the model is genuinely uncertain about **out-of-sample** cluster membership — precisely the points where a hard assignment would be least trustworthy, and where the underlying soft/conditional-expectation view (rather than a hard label) is most informative.
